In [1]:
import numpy as np
from scipy.stats import gamma

# HRF
t = np.arange(0, 32, 1)
hrf = gamma.pdf(t, 6) - 0.35 * gamma.pdf(t, 16)
hrf /= hrf.max()

n_timepoints = 336  # your actual number of volumes per run
n_simulations = 1000
true_prep_amplitude = 1.0
true_exec_amplitude = 3.0  # typically larger

bias_glm12 = []
bias_glm16 = []

for sim in range(n_simulations):
    # Build true signal using your actual trial timing
    # (replace with your real cue onset and perturbation onset times)
    prep_delta = np.zeros(n_timepoints)
    exec_delta = np.zeros(n_timepoints)
    prep_boxcar = np.zeros(n_timepoints)
    
    # Example timing - replace with your actual onsets
    cue_onsets = [10, 40, 70, 100, 130]  # in TRs
    exec_onsets = [12, 42, 72, 102, 132]  # ~2s later
    prep_durations = [2, 2, 2, 2, 2]  # in TRs
    
    for onset in cue_onsets:
        prep_delta[onset] = 1
    for onset, dur in zip(cue_onsets, prep_durations):
        prep_boxcar[onset:onset+dur] = 1
    for onset in exec_onsets:
        exec_delta[onset] = 1
    
    # Convolve with HRF
    prep_delta_conv = np.convolve(prep_delta, hrf)[:n_timepoints]
    prep_boxcar_conv = np.convolve(prep_boxcar, hrf)[:n_timepoints]
    exec_conv = np.convolve(exec_delta, hrf)[:n_timepoints]
    
    # True signal
    noise = np.random.normal(0, 0.5, n_timepoints)
    true_signal = (true_prep_amplitude * prep_delta_conv + 
                   true_exec_amplitude * exec_conv + noise)
    
    # Fit GLM 12 (delta for prep)
    X12 = np.column_stack([prep_delta_conv, exec_conv, np.ones(n_timepoints)])
    beta12 = np.linalg.lstsq(X12, true_signal, rcond=None)[0]
    bias_glm12.append(beta12[0] - true_prep_amplitude)
    
    # Fit GLM 16 (boxcar for prep)
    X16 = np.column_stack([prep_boxcar_conv, exec_conv, np.ones(n_timepoints)])
    beta16 = np.linalg.lstsq(X16, true_signal, rcond=None)[0]
    bias_glm16.append(beta16[0] - true_prep_amplitude)

print(f'GLM 12 mean bias: {np.mean(bias_glm12):.4f}')
print(f'GLM 16 mean bias: {np.mean(bias_glm16):.4f}')
print(f'GLM 12 std of bias: {np.std(bias_glm12):.4f}')
print(f'GLM 16 std of bias: {np.std(bias_glm16):.4f}')

GLM 12 mean bias: 0.0016
GLM 16 mean bias: -0.3503
GLM 12 std of bias: 0.1799
GLM 16 std of bias: 0.1171
